# **Machine learning: Recognition of (magic) spells**

## 1. Introduction

### a) Review

This notebook follows the Perceptron practical session in which we built a small neural network - called a **perceptron** - to perform character recognition.


<img src="https://github.com/PlantecJY/python-mic/raw/master/principe.png" alt="Drawing" width="300" align="center"/>


We followed the steps below:
* **Data structuring** for training (a dictionary of written digits represented by 5*6 arrays, as in the example below

```
nb9b = [[1,1,1,1,1],
        [1,0,0,0,1],
        [1,0,0,0,1],
        [1,1,1,1,1],
        [0,0,0,0,1],
        [1,1,1,1,1]]
```
* **Creation of the neural network**: an input layer of dimension 300 and an output layer of dimension 10; the information passed to each output neuron was simply the linear combination of the neurons connected to this output weighted by the weight of each neuron;
* **Testing** the operation of the neural network;
* **Training**, that is, optimization of the weight of each neuron;
* Using the neural network to make a **prediction**: a digit is provided as an array and the network is asked to predict this digit.

### b) Objective of this practical session

In this practical session, the objective is to understand the entire **Machine Learning training pipeline** through the field of **gesture recognition**, and more particularly (magic) spells. We will therefore go through the following steps:
  1) Raw data acquisition and construction of a **dataset**
  2) **Split** training data/test data
  3) **Pre-processing** of the data (feature generation)
  4) **Training** a lightweight model (neural network)
  5) **Testing** the model
  6) **Embedded deployment** on a board


### c) Steps to follow

1) **PART 1**: Understand the entire training pipeline through an already prepared dataset
2) **PART 2**: Embedded deployment of the model on the board
3) **PART 3**: Complete the database: add a new gesture and repeat the training pipeline through to deployment.


## **PART 1: Understanding the machine learning training pipeline**

## 1. Building a dataset

For this first step, we are going to download a database already containing a certain number of gestures per **label** (there are 3):
* 100 gestures of the "**Wingardium Leviosa**" spell
* 100 gestures of the "**Stupefy**" spell
* 100 gestures of the "**Expelliarmus**" spell

The gestures are recorded, with the board in hand, for a duration of **5 seconds**; it is important to respect this acquisition time when recording new data.

The objective here is to build a Python table containing the data that we can then process to train/test the future model.


### a) Importing the necessary files/libraries

In [ ]:
#Once this line has been executed, Restart Kernel

%pip install pyserial

In [ ]:
import zipfile
import os
import glob
import json
import pandas as pd
import os

### b) Questions

**Open the Moodle quiz**

1) What is the size of the dataset? (Number of lines) Use the .shape function.
2) What does the data_sort file contain?
4) What is the data recording frequency? Use the size of the dataset, the total number of recordings and the number of seconds in one recording. Round to the nearest integer.
5) What is the value of the gyroscope Z axis in the first row of the data table? Use the head function.
6) Why is it important to check data quality before starting to train a model?
7) Match these concepts to their definitions: Label, Acquisition, Dataset, Feature

### c) Uploading a dataset and organizing the data in a table

Note: here, the data are said to be "**raw**"; they have not yet been processed to train and test the model. Here we have the time-series acquisitions for each gesture recording, which gives many more rows in the table than actual files.

In [ ]:
data = pd.read_csv("dataset_sorts_xiao.csv")

colonnes_attendues = [
    "AxX", "AxY", "AxZ",
    "GxX", "GxY", "GxZ",
    "source_file", "label", "origine",
]

colonnes_manquantes = set(colonnes_attendues) - set(data.columns)
if colonnes_manquantes:
    raise ValueError(f"Missing columns: {colonnes_manquantes}")

display(
    data.groupby("label")["source_file"]
    .nunique()
    .rename("number_of_gestures")
    .to_frame()
)

#Your code here:


## 2. Splitting the data into training/test sets

In the second step of the training pipeline, we are going to **separate (split)** the data into 2 categories:
* The **training data** (training set), which will be used to train the model; these data will themselves be divided into model training and validation data
* The **test data** (test set), which will be used to test the model once it has been trained

<img src="https://raw.githubusercontent.com/adarshbk12/Stage_IA/master/split.jpg" alt="Drawing" width="300" align="center"/>

### a) Importing the necessary files/libraries

In [ ]:
import split_data

### b) Questions

**Back to the Moodle quiz**

1) Perform the split with 25% test data, then give the number of rows in the test data.
4) Why shuffle the data before the split?
5) Why is it important to have data that the model has never seen?
6) Should the test set be used to train the model?
8) If we have 300 acquisitions and an 80/20 split, how many acquisitions are used for training?
9) Match these concepts to their definitions: Training, validation, test

### c) Performing the split

In [ ]:
#Example: 40% of the data for testing
#train_data, test_data = split_data.split_dataset(data, test_size=0.40)

#Your code here for a 75/25 split:


### d) Graph of the data distribution

In [ ]:
split_data.plot_data_split(train_data, test_data)

### e) Graph of the data distribution by LABEL

In [ ]:
split_data.plot_label_split(train_data, test_data)

## 3. Spectral analysis of one data sample

In this section, we are going to perform the **spectral analysis** of one particular data sample. We will first observe its **time-domain representation** and then analyze it using a **Fast Fourier Transform (FFT)**. This will then allow us to generate characteristics (**features**) for all the data that can be used to train and test the model.


### a) Importing the necessary files/libraries

In [ ]:
import analyse_spec

### b) Questions

**Back to the Moodle quiz**

1) Why is it preferable to perform a frequency analysis rather than a time-domain analysis in this case?
2) What type of filter would you apply after observing the spectral analysis of Expelliarmus signal number 0?
6) For "Wingardium Leviosa" spell number 2, which IMU axis is predominant around 2 Hz?
7) How does noise affect the frequency spectrum?

### c) Time-domain representation of the raw data

In [ ]:
#Choice of a function to analyze

# Example of using the choisir_acquisition function

sample = analyse_spec.choisir_acquisition(train_data, "Stupefy", 10)

#Your code here:



In [ ]:
# Time-domain representation of the raw data (accelerometer)

analyse_spec.afficher_signaux_temporels(sample)

### d) Spectral analysis of the data

In [ ]:
# Spectral analysis (FFT)

resultats_fft = analyse_spec.afficher_analyse_spectrale(
    sample,
    interval_ms=20,
)

## 4. Data pre-processing and feature generation

Here, we are going to perform the **pre-processing of the data** before training the model:
* We will first perform a **split of the training data** into 2: those that will be used to train the model (**training set**) and those that will be used to validate it and tune the hyperparameters (**validation set**).
* Next, we will perform the spectral analysis of all the data to allow the **generation of characteristics** (features) that will be used for training and testing.
* We will also perform some processing on the features before using them.

### a) Importing the necessary files/libraries

In [ ]:
import pretraitement

### b) Questions

**Back to the Moodle quiz**

1) Perform the train/validation split at 80/20; how many rows are there for the validation data?
2) What is the difference between test data and validation data?
3) (CODE, not a Moodle question) Generate the features for the 2 validation and test datasets by following the example for "features_train": use the names "features_validation" and "features_test".
5) Why is it important to pre-process the data before training?
6) Why is it important to normalize and encode the features before training?
9) Match each of these methods to its definition: Hold-Out, K-Fold and Leave-One-Out

### c) Train/validation split

In [ ]:
#Function that performs the train/validation split

# Example of second split: Train / Validation at 60/40
# train_data, validation_data = pretraitement.split_train_validation(
#    train_data,
#    validation_size=0.4)

#Your code here:


### d) Generating characteristics for all the data

In [ ]:
# Feature generation for the three datasets
features_train = pretraitement.construire_features(train_data)

#Your code here:


### e) Encoding and normalization of the features

Here, before training the model, it is important to **encode and normalize** the characteristics to allow optimal and efficient training.

In [ ]:
#Encoding and normalization of the features

donnees_preparees = pretraitement.preparer_donnees_modele(
    features_train,
    features_validation,
    features_test,
)

X_train = donnees_preparees["X_train"]
X_validation = donnees_preparees["X_validation"]
X_test = donnees_preparees["X_test"]

y_train = donnees_preparees["y_train"]
y_validation = donnees_preparees["y_validation"]
y_test = donnees_preparees["y_test"]

X_train_scaled = donnees_preparees["X_train_scaled"]
X_validation_scaled = donnees_preparees["X_validation_scaled"]
X_test_scaled = donnees_preparees["X_test_scaled"]

y_train_encoded = donnees_preparees["y_train_encoded"]
y_validation_encoded = donnees_preparees["y_validation_encoded"]
y_test_encoded = donnees_preparees["y_test_encoded"]

label_encoder = donnees_preparees["label_encoder"]
scaler = donnees_preparees["scaler"]

## 5. Classification on training data

We now move on to the **model training** part with **a neural network**, using the training data.

<img src="https://raw.githubusercontent.com/adarshbk12/Stage_IA/master/reseau_neurones.png"
     alt="Neural network"
     width="600">

This will then allow us to observe the **model performance** (metrics, confusion matrix) with the validation data, which will be used to tune and adjust the model.

### a) Importing the necessary files

In [ ]:
import entrainement

### b) Questions

**Back to the Moodle quiz**

1) How many layers does the network contain? Analyze the neural network architecture.
2) What activation function is used for the neural network output?
3) Compile and train the model. What does the number of epochs have an effect on?
4) (CODE, not a Moodle question) Display the network loss function with the "entrainement.afficher_loss()" function; follow the example of the function used to display accuracy.
5) Match each metric to its definition.
6) (CODE, not a Moodle question) Display the confusion matrix with "entrainement.afficher_matrice_confusion"; if necessary, look at its definition in the dedicated file.
8) How can the detection probability and false-positive ratio be obtained from the confusion matrix?
9) What is the role of the activation function?
10) How can overfitting be detected from the training and validation curves?

### c) Creating the neural network

In [ ]:
#Neural network architecture

num_features = X_train_scaled.shape[1]
num_classes = len(label_encoder.classes_)

model = entrainement.Sequential([
    entrainement.Dense(64, activation="relu", input_shape=(num_features,)),
    entrainement.Dropout(0.3),

    entrainement.Dense(32, activation="relu"),
    entrainement.Dropout(0.3),

    entrainement.Dense(num_classes, activation="softmax")
])

### d) Compiling the model

In [ ]:
# Model compilation

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### e) Training the model

In [ ]:
#Training 

early_stop = entrainement.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train_encoded,
    validation_data=(X_validation_scaled, y_validation_encoded),
    epochs=10,
    batch_size=16,
    callbacks=[early_stop]
)

### f) Training performance

In [ ]:
#Accuracy graph

entrainement.afficher_accuracy(history)

In [ ]:
#Graph of the model loss function

#Your code here:


In [ ]:
# Predictions on the validation set
y_validation_proba = model.predict(X_validation_scaled)
y_validation_pred = entrainement.np.argmax(y_validation_proba, axis=1)

#Metrics

metrics_validation = entrainement.afficher_metriques_classification(
    y_validation_encoded,
    y_validation_pred,
    y_validation_proba,
    titre="Metrics - Validation",
)

In [ ]:
#Confusion matrix

#Your code here:


## 6. Testing the model

For this final step before embedded deployment of the model on the board, we are going to **test the model** with data it has never seen, the **test data**. This test will provide the **real performance of the model** when it is presented with new data that were not used to train it.

### a) Importing the necessary files/libraries

In [ ]:
import entrainement

### b) Questions

**Back to the Moodle quiz**

1) (CODE, not a Moodle question) Display the confusion matrix.
4) Why is test accuracy lower than validation accuracy during training?
5) What do false positives represent? And false negatives?
6) Does an accuracy of 100% guarantee a perfectly reliable model?

### c) Encoding and normalization of the features

Here, before testing the model, it is important to **encode and normalize** the characteristics to allow optimal and efficient testing.

In [ ]:
# Test data

X_test = features_test.drop(
    columns=["source_file", "label"]
)

y_test = features_test["label"]

# Label encoding
y_test_encoded = label_encoder.transform(y_test)

# Feature normalization
X_test_scaled = scaler.transform(X_test)

### d) Model prediction on the test data

In [ ]:
# Predictions on the test set
y_test_proba = model.predict(X_test_scaled)
y_test_pred = entrainement.np.argmax(y_test_proba, axis=1)

### e) Model performance

In [ ]:
metrics_test = entrainement.afficher_metriques_classification(
    y_test_encoded,
    y_test_pred,
    y_test_proba,
    titre="Metrics - Test",
)

In [ ]:
#Confusion matrix

#Your code here:


## **PART 2: Deploying the model on the Seeed XIAO nRF52840 Sense board**

We are going to embed the neural network on the Seeed Xiao NRF52840 Sense board. The notebook **simply exports the learned weights and biases into a C++ file** (`model_params.h`). On the board, a C++ function directly computes the dense layers, the ReLU activations and then the softmax.

The procedure for direct acquisition on the board is then as follows:

1. acquisition of IMU measurements at 50 Hz (5 s window);
2. calculation of the same characteristics as during training;
3. normalization with the `StandardScaler` parameters;
4. C++ inference: dense layer 138→32, ReLU, dense layer 32→16, ReLU, dense layer 16→classes, softmax;
5. display of the class and LED blinking.

**Here are the steps to follow**:
* Connect the magic wand board to the computer via the USB cable
* IT IS IMPORTANT THAT THE WAND POWER SUPPLY IS IN OFF MODE
* Then carry out the following steps to load the model onto the board and test it without connecting it to the computer

### a) Presentation of the Seeed Xiao NRF52840 Sense board

<img src="https://raw.githubusercontent.com/adarshbk12/Stage_IA/master/Seeed-Studio-XIAO-nRF52840-Sense.jpg" width="300">

The **Seeed Xiao NRF52840 Sense** board is equipped with an **IMU** (Inertial Measurement Unit), which returns data from a **gyroscope** (on 3 axes) and an **accelerometer** (on 3 axes); this will allow us to analyze the gestures performed.

We are therefore going to load the model we trained onto this board; in PART 3, we will then record new data using it.

### b) Importing the necessary files/libraries

In [ ]:
import deploiement

### c) Questions

**Back to the Moodle quiz**

1) Why do we talk about an "embedded" model?
2) What does the `model_params.h` file contain?
3) Must the same pre-processing be applied in Python and on the board?
8) What memory and computation-time constraints can be encountered?
9) Why must the actual acquisition rate remain close to 50 Hz?

### d) Installing support for the Seeed Xiao NRF52840 Sense board

In [ ]:
ARDUINO_CLI = deploiement.installer_support_xiao_nrf52840()

### e) Installing the necessary Arduino libraries

In [ ]:
deploiement.installer_bibliotheque_lsm6ds3(
    ARDUINO_CLI
)

### f) Exporting the model parameters

In [ ]:
deploiement.exporter_modele_cpp(
    model,
    scaler,
    label_encoder,
)

### g) Deploying the model on the board

Here, we are going to generate an **Arduino code containing the model** that will be implemented on the board.

In [ ]:
# ============================================================
# Generation, compilation and upload of the Arduino sketch
# Direct inference in C++: Dense + ReLU + Softmax
# ============================================================

informations_deploiement = deploiement.deployer_modele_xiao(
    arduino_code=deploiement.arduino_code,
    arduino_cli=ARDUINO_CLI,
    fichier_parametres="model_params.h",
)

## **Now test the model by unplugging the board and turning on the power supply! (Have spell battles with the magic wand :))**

## **PART 3: Adding a new gesture through fine-tuning**

This part performs the entire system improvement loop:

1. temporarily replace the embedded model with acquisition firmware;
2. record several examples of a new label for 5 seconds at 50 Hz;
3. adapt the model already trained in part 1 without starting again from random weights;
4. export the new weights to C++ and re-embed the model as in part 2.

Fine-tuning consists of taking a model already trained on the old gestures and training it a little more with the new gesture.

The model thus retains what it has already learned, while adapting its parameters to recognize an additional class. During this step, it is also necessary to present it with examples of the old gestures so that it does not forget them.

Fine-tuning reuses the pre-processing, the `StandardScaler` and the hidden layers from part 1. The output layer is enlarged in order to add the new label. The old data are mixed with the new data to limit forgetting of the spells already known.

## 1. Loading the acquisition firmware onto the board

The classification model is temporarily replaced by a simple program. When Python sends the `S` command, the board returns 250 IMU measurements, i.e. 5 seconds at 50 Hz, then sends `FIN`.

The three LEDs remain on during acquisition.

### a) Importing the necessary files/libraries

In [ ]:
import nouveau_label

### b) Uploading the acquisition firmware

In [ ]:
# Uploading the firmware for new-data acquisition

PORT = nouveau_label.charger_firmware_acquisition_xiao()

## 2. Recording the new label

Choose **Create a new label**, then ideally record at least 20 acquisitions. Always perform the same gesture, with natural variations in speed, orientation and amplitude.

### a) Importing the necessary files/libraries

In [ ]:
import nouvelle_acquisition
import deploiement

### b) Questions about acquisition

**Back to the Moodle quiz**

1. Why must a 5-second window and a frequency of 50 Hz be maintained?
2. How many gestures are ideally needed for the new label?
3. Why should speed and orientation be varied slightly without changing the nature of the gesture?

### c) Launching the acquisition menu

In [ ]:
data = nouvelle_acquisition.menu_acquisition(
    dataset=data,
    fonction_trouver_port=nouvelle_acquisition.trouver_port_xiao,
)


## 3. Fine-tuning the model from part 1

This step does not recreate a network from scratch:

- the two hidden layers and their weights are reused from the model in part 1;
- the output layer is enlarged;
- the output weights of the old labels are copied by name;
- the new output neuron is initialized;
- the head is first trained alone, then the entire network is adjusted with a low learning rate.

The scaler from part 1 is retained: recalculating it would change the numerical meaning of the inputs and make the old weights inconsistent.

In [ ]:
import importlib
import nouveau_label
import pretraitement

importlib.reload(nouveau_label)

In [ ]:
model, scaler, label_encoder, resultats_finetuning = (
    nouveau_label.fine_tuner_avec_nouveau_label(
        model_base=model,
        scaler_base=scaler,
        label_encoder_base=label_encoder,
        dataset=data,
        fonction_construire_features=pretraitement.construire_features,
        epochs_tete=10,
        epochs_completes=20,
        batch_size=16,
        random_state=42,
    )
)

print(
    "New label(s):",
    resultats_finetuning["labels_ajoutes"],
)

print(
    "Embedded classes:",
    list(label_encoder.classes_),
)

## 4. Re-embedding the adapted model

The following cell regenerates `model_params.h` with the additional class, then repeats exactly the compilation, upload and serial monitor steps from part 2.

After uploading, the serial monitor legend must contain the new label and its number of blinks.

In [ ]:
import importlib
import nouveau_label
import deploiement

importlib.reload(nouveau_label)

In [ ]:
resultats_deploiement = (
    nouveau_label.exporter_et_deployer_modele_xiao(
        model=model,
        scaler=scaler,
        label_encoder=label_encoder,
        arduino_code=nouveau_label.arduino_code,

        fonction_exporter=deploiement.exporter_modele_cpp,
        fonction_deployer=deploiement.deployer_modele_xiao,

        arduino_cli=ARDUINO_CLI,
        fichier_parametres="model_params.h",
        ouvrir_moniteur=True,
    )
)

## Unplug the board and grab your wands, wizards!

# **END OF THE PRACTICAL SESSION**